<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  📚 Importing Libraries
</h2>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🔄 Load the <code>.npy</code> File and Use a Boolean Value to Determine Whether to Perform Classification or Regression
</h2>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

x_train = np.load('/content/drive/MyDrive/Time Series Data/Blink/x_train.npy')
y_train = np.load('/content/drive/MyDrive/Time Series Data/Blink/y_train.npy')
x_valid = np.load('/content/drive/MyDrive/Time Series Data/Blink/x_test.npy')
y_valid = np.load('/content/drive/MyDrive/Time Series Data/Blink/y_test.npy')

is_classification = len(np.unique(y_train)) / len(y_train) < 0.05
print("Classification task:", is_classification)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Classification task: True


<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🏷️ Generating Numerical Class Labels for Classification
</h2>

In [ ]:
if is_classification:
    print(y_train[:10])
    from sklearn.preprocessing import LabelEncoder

    if not np.issubdtype(y_train.dtype, np.number):
        le = LabelEncoder()
        y_train = le.fit_transform(y_train)
        y_valid = le.transform(y_valid)
        print("Labels were non-numeric. Encoded classes:", le.classes_)
    else:
        print("Labels are already numeric. No encoding needed.")
    print(y_train[:10])

['longblink' 'longblink' 'longblink' 'longblink' 'longblink' 'longblink'
 'longblink' 'longblink' 'longblink' 'longblink']
Labels were non-numeric. Encoded classes: ['longblink' 'shortblink']
[0 0 0 0 0 0 0 0 0 0]


<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🧊 Tensor & Dimension
</h2>

In [ ]:
print("x_train: ", x_train.shape)
print("y_train: ", y_train.shape)
print("x_valid: ", x_valid.shape)
print("y_valid: ", y_valid.shape)

x_train = torch.tensor(x_train, dtype=torch.float32)
x_valid = torch.tensor(x_valid, dtype=torch.float32)

if is_classification:
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_valid = torch.tensor(y_valid, dtype=torch.long)
else:
    y_train = torch.tensor(y_train, dtype=torch.float32)
    y_valid = torch.tensor(y_valid, dtype=torch.float32)

x_train:  (500, 20000)
y_train:  (500,)
x_valid:  (450, 20000)
y_valid:  (450,)


<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  📡 Bringing Channel
</h2>

In [ ]:
if x_train.ndim == 2:
    x_train = x_train.unsqueeze(1)
    x_valid = x_valid.unsqueeze(1)

print(x_train.shape)
print(x_valid.shape)
print(f"Train samples  : {x_train.shape[0]}")
print(f"Test samples   : {x_valid.shape[0]}")
print(f"Channels       : {x_train.shape[1]}")
print(f"Signal length  : {x_train.shape[2]}")

torch.Size([500, 1, 20000])
torch.Size([450, 1, 20000])
Train samples  : 500
Test samples   : 450
Channels       : 1
Signal length  : 20000


<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  ⚖️ Normalization
</h2>

In [ ]:
mean = x_train.mean(dim=-1, keepdim=True)
std  = x_train.std(dim=-1, keepdim=True) + 1e-6
x_train = (x_train - mean) / std

mean = x_valid.mean(dim=-1, keepdim=True)
std  = x_valid.std(dim=-1, keepdim=True) + 1e-6
x_valid = (x_valid - mean) / std

print("After normalization:")
print("x_train mean:", x_train.mean().item(), "std:", x_train.std().item())
print("x_valid mean:", x_valid.mean().item(), "std:", x_valid.std().item())

After normalization:
x_train mean: -1.4314342600130203e-08 std: 0.99997478723526
x_valid mean: 8.278145280371518e-09 std: 0.99997478723526


<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🤖 CNN Model Class for Both
</h2>

In [ ]:
class Classifier(nn.Module):
    def __init__(self, c_in, n_outputs, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(c_in, 32, kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
        )
        self.block1 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(4),
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5, padding='same'),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(4),
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=3, padding='same'),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(4),
        )
        self.block4 = nn.Sequential(
            nn.Conv1d(256, 256, kernel_size=3, padding='same'),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, n_outputs),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.head(x)

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🏗️ Building Model
</h2>

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if is_classification:
    n_outputs = len(torch.unique(y_train))
else:
    n_outputs = 1

print("n_outputs:", n_outputs)

model = Classifier(c_in=1, n_outputs=n_outputs).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Learnable parameters: {total_params/1e6:.3f}M")

with torch.no_grad():
    dummy = torch.randn(2, 1, x_train.shape[-1]).to(device)
    print("Output shape:", model(dummy).shape)

Device: cuda
n_outputs: 1
Classifier(
  (stem): Sequential(
    (0): Conv1d(1, 32, kernel_size=(15,), stride=(4,), padding=(7,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (block1): Sequential(
    (0): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=same)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=same)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=same)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  📦 Data Loader
</h2>

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

train_loader = DataLoader(
    TensorDataset(x_train, y_train),
    batch_size=batch_size, shuffle=True, drop_last=False,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    TensorDataset(x_valid, y_valid),
    batch_size=batch_size, shuffle=False, drop_last=False,
    pin_memory=(device.type == "cuda"),
)

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🔁 Training Loop
</h2>

In [ ]:
epochs    = 25
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

if not is_classification:
    y_std = y_train.std().item()


def evaluate(model, loader):
    """Returns (loss, metric).
    metric = accuracy for classification, NRMSE for regression."""
    model.eval()
    total_samples = 0
    total_loss    = 0.0

    if is_classification:
        total_correct = 0
    else:
        total_sq_err = 0.0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out  = model(x)

            if is_classification:
                loss   = F.cross_entropy(out, y)
                preds  = torch.argmax(out, dim=1)
                total_correct += (preds == y).sum().item()
            else:
                preds = out.squeeze(1)
                loss  = F.mse_loss(preds, y)
                total_sq_err += ((preds - y) ** 2).sum().item()

            bs             = y.size(0)
            total_samples += bs
            total_loss    += loss.item() * bs

    avg_loss = total_loss / total_samples
    if is_classification:
        metric = total_correct / total_samples
    else:
        rmse   = (total_sq_err / total_samples) ** 0.5
        metric = rmse / y_std
    return avg_loss, metric


for epoch in range(epochs):
    model.train()
    epoch_loss    = 0.0
    total_samples = 0

    if is_classification:
        epoch_correct = 0
    else:
        epoch_sq_err = 0.0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out  = model(x)

        if is_classification:
            loss = F.cross_entropy(out, y)
        else:
            preds = out.squeeze(1)
            loss  = F.mse_loss(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs             = y.size(0)
        epoch_loss    += loss.item() * bs
        total_samples += bs

        if is_classification:
            epoch_correct += (torch.argmax(out.detach(), dim=1) == y).sum().item()
        else:
            epoch_sq_err += ((preds.detach() - y) ** 2).sum().item()

    scheduler.step()

    train_loss = epoch_loss / total_samples
    if is_classification:
        train_metric = epoch_correct / total_samples
    else:
        train_metric = ((epoch_sq_err / total_samples) ** 0.5) / y_std

    val_loss, val_metric = evaluate(model, val_loader)

    if is_classification:
        print(
            f"Epoch {epoch+1:2d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_metric:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_metric:.4f}"
        )
    else:
        print(
            f"Epoch {epoch+1:2d} | "
            f"Train Loss: {train_loss:.4f} | Train NRMSE: {train_metric:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val NRMSE: {val_metric:.4f}"
        )

train_loss, train_metric = evaluate(model, train_loader)
val_loss,   val_metric   = evaluate(model, val_loader)

print("\nFinal Results")
if is_classification:
    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_metric*100:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_metric*100:.2f}%")
else:
    print(f"Train Loss: {train_loss:.4f} | Train NRMSE: {train_metric:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val NRMSE:   {val_metric:.4f}")

Epoch  1 | Train Loss: 14686.5535 | Train NRMSE: 2.9150 | Val Loss: 14064.9637 | Val NRMSE: 2.8526
Epoch  2 | Train Loss: 12894.6871 | Train NRMSE: 2.7314 | Val Loss: 11860.5258 | Val NRMSE: 2.6195
Epoch  3 | Train Loss: 10682.4672 | Train NRMSE: 2.4860 | Val Loss: 9342.7608 | Val NRMSE: 2.3249
Epoch  4 | Train Loss: 8353.4942 | Train NRMSE: 2.1984 | Val Loss: 6716.2200 | Val NRMSE: 1.9712
Epoch  5 | Train Loss: 6253.9342 | Train NRMSE: 1.9022 | Val Loss: 5205.9768 | Val NRMSE: 1.7355
Epoch  6 | Train Loss: 4568.6728 | Train NRMSE: 1.6258 | Val Loss: 3571.3897 | Val NRMSE: 1.4374
Epoch  7 | Train Loss: 3397.7753 | Train NRMSE: 1.4021 | Val Loss: 2998.9716 | Val NRMSE: 1.3172
Epoch  8 | Train Loss: 2625.0619 | Train NRMSE: 1.2324 | Val Loss: 2186.6859 | Val NRMSE: 1.1248
Epoch  9 | Train Loss: 2230.7012 | Train NRMSE: 1.1360 | Val Loss: 2903.4439 | Val NRMSE: 1.2961
Epoch 10 | Train Loss: 2012.9090 | Train NRMSE: 1.0792 | Val Loss: 1828.6133 | Val NRMSE: 1.0286
Epoch 11 | Train Loss: 18

<h3><img src="https://cdn-icons-png.flaticon.com/512/2103/2103832.png" width="24" height="24" style="vertical-align: middle; margin-right: 8px;"> RNN / LSTM / GRU</h3>

<h3><img src="https://img.icons8.com/color/48/000000/data-configuration.png" width="26" height="26" style="vertical-align: middle; margin-right: 8px;"> Loading Data and Encoding</h3>

In [ ]:
x_train_np = np.load('/content/drive/MyDrive/Time Series Data/Blink/x_train.npy')
y_train_np = np.load('/content/drive/MyDrive/Time Series Data/Blink/y_train.npy')
x_valid_np = np.load('/content/drive/MyDrive/Time Series Data/Blink/x_test.npy')
y_valid_np = np.load('/content/drive/MyDrive/Time Series Data/Blink/y_test.npy')

if is_classification and not np.issubdtype(y_train_np.dtype, np.number):
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_train_np = le.fit_transform(y_train_np)
    y_valid_np = le.transform(y_valid_np)

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🧊 Tensor & Dimension
</h2>

In [ ]:
x_train_rnn = torch.tensor(x_train_np, dtype=torch.float32)
x_valid_rnn = torch.tensor(x_valid_np, dtype=torch.float32)

if is_classification:
    y_train_rnn = torch.tensor(y_train_np, dtype=torch.long)
    y_valid_rnn = torch.tensor(y_valid_np, dtype=torch.long)
else:
    y_train_rnn = torch.tensor(y_train_np, dtype=torch.float32)
    y_valid_rnn = torch.tensor(y_valid_np, dtype=torch.float32)

x_train_rnn = x_train_rnn.unsqueeze(1)
x_valid_rnn = x_valid_rnn.unsqueeze(1)

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  📉 Normalization and Downsample
</h2>

In [ ]:
mean = x_train_rnn.mean(dim=-1, keepdim=True)
std  = x_train_rnn.std(dim=-1, keepdim=True) + 1e-6
x_train_rnn = (x_train_rnn - mean) / std

mean = x_valid_rnn.mean(dim=-1, keepdim=True)
std  = x_valid_rnn.std(dim=-1, keepdim=True) + 1e-6
x_valid_rnn = (x_valid_rnn - mean) / std

x_train_rnn = F.avg_pool1d(x_train_rnn, kernel_size=40)
x_valid_rnn = F.avg_pool1d(x_valid_rnn, kernel_size=40)

print("Shapes after prep:")
print("  x_train_rnn:", x_train_rnn.shape)
print("  x_valid_rnn:", x_valid_rnn.shape)

Shapes after prep:
  x_train_rnn: torch.Size([3493, 1, 500])
  x_valid_rnn: torch.Size([1510, 1, 500])


<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🗂️ Classifier for RNN / LSTM / GRU
</h2>

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, c_in, n_outputs, hidden_size=128, num_layers=2,
                 rnn_type='gru', dropout=0.3):
        super().__init__()
        rnn_cls = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[rnn_type]
        self.rnn_type = rnn_type

        self.rnn = rnn_cls(
            input_size=c_in,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_outputs),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out = self.rnn(x)
        if self.rnn_type == 'lstm':
            output, (hn, cn) = out
        else:
            output, hn = out
        return self.head(hn[-1])

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🤖 Choose Model
</h2>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

if is_classification:
    n_outputs_rnn = len(torch.unique(y_train_rnn))
else:
    n_outputs_rnn = 1

selector = widgets.ToggleButtons(
    options=['rnn', 'lstm', 'gru'],
    value='gru',
    description='RNN type:',
)
output_area = widgets.Output()

rnn_model = None

def build_model(rnn_type):
    global rnn_model
    with output_area:
        clear_output()
        print(f"Using: {rnn_type.upper()}")
        rnn_model = RNNClassifier(
            c_in=1, n_outputs=n_outputs_rnn,
            hidden_size=128, num_layers=2,
            rnn_type=rnn_type,
        ).to(device)

        total_params = sum(p.numel() for p in rnn_model.parameters() if p.requires_grad)
        print(rnn_model)
        print(f"Learnable parameters: {total_params/1e6:.3f}M")

        with torch.no_grad():
            dummy = torch.randn(2, 1, x_train_rnn.shape[-1]).to(device)
            print("Output shape:", rnn_model(dummy).shape)

def on_change(change):
    if change['name'] == 'value':
        build_model(change['new'])

selector.observe(on_change)
display(selector, output_area)
build_model(selector.value)

ToggleButtons(description='RNN type:', index=2, options=('rnn', 'lstm', 'gru'), value='gru')

Output()

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  📦 Data Loader
</h2>

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

train_loader_rnn = DataLoader(
    TensorDataset(x_train_rnn, y_train_rnn),
    batch_size=batch_size, shuffle=True, drop_last=False,
    pin_memory=(device.type == "cuda"),
)

val_loader_rnn = DataLoader(
    TensorDataset(x_valid_rnn, y_valid_rnn),
    batch_size=batch_size, shuffle=False, drop_last=False,
    pin_memory=(device.type == "cuda"),
)

<h2 style="color:#4285F4; border-bottom: 2px solid #4285F4; padding-bottom: 8px;">
  🔁 Training Loop
</h2>

In [ ]:
epochs    = 25
optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

if not is_classification:
    y_std_rnn = y_train_rnn.std().item()

model_name = rnn_model.rnn_type.upper()
print(f"Training {model_name} model for {epochs} epochs\n")


def evaluate_rnn(model, loader):
    model.eval()
    total_samples = 0
    total_loss    = 0.0
    if is_classification:
        total_correct = 0
    else:
        total_sq_err = 0.0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out  = model(x)

            if is_classification:
                loss  = F.cross_entropy(out, y)
                preds = torch.argmax(out, dim=1)
                total_correct += (preds == y).sum().item()
            else:
                preds = out.squeeze(1)
                loss  = F.mse_loss(preds, y)
                total_sq_err += ((preds - y) ** 2).sum().item()

            bs             = y.size(0)
            total_samples += bs
            total_loss    += loss.item() * bs

    avg_loss = total_loss / total_samples
    if is_classification:
        metric = total_correct / total_samples
    else:
        rmse   = (total_sq_err / total_samples) ** 0.5
        metric = rmse / y_std_rnn
    return avg_loss, metric


for epoch in range(epochs):
    rnn_model.train()
    epoch_loss    = 0.0
    total_samples = 0
    if is_classification:
        epoch_correct = 0
    else:
        epoch_sq_err = 0.0

    for x, y in train_loader_rnn:
        x, y = x.to(device), y.to(device)
        out  = rnn_model(x)

        if is_classification:
            loss = F.cross_entropy(out, y)
        else:
            preds = out.squeeze(1)
            loss  = F.mse_loss(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs             = y.size(0)
        epoch_loss    += loss.item() * bs
        total_samples += bs

        if is_classification:
            epoch_correct += (torch.argmax(out.detach(), dim=1) == y).sum().item()
        else:
            epoch_sq_err += ((preds.detach() - y) ** 2).sum().item()

    scheduler.step()

    train_loss = epoch_loss / total_samples
    if is_classification:
        train_metric = epoch_correct / total_samples
    else:
        train_metric = ((epoch_sq_err / total_samples) ** 0.5) / y_std_rnn

    val_loss, val_metric = evaluate_rnn(rnn_model, val_loader_rnn)

    if is_classification:
        print(
            f"Epoch {epoch+1:2d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_metric:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_metric:.4f}"
        )
    else:
        print(
            f"Epoch {epoch+1:2d} | "
            f"Train Loss: {train_loss:.4f} | Train NRMSE: {train_metric:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val NRMSE: {val_metric:.4f}"
        )

train_loss, train_metric = evaluate_rnn(rnn_model, train_loader_rnn)
val_loss,   val_metric   = evaluate_rnn(rnn_model, val_loader_rnn)

print(f"\nFinal Results ({model_name})")
if is_classification:
    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_metric*100:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_metric*100:.2f}%")
else:
    print(f"Train Loss: {train_loss:.4f} | Train NRMSE: {train_metric:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val NRMSE:   {val_metric:.4f}")

Training GRU model for 25 epochs

Epoch  1 | Train Loss: 13765.6976 | Train NRMSE: 2.8221 | Val Loss: 12703.6494 | Val NRMSE: 2.7110
Epoch  2 | Train Loss: 11827.3631 | Train NRMSE: 2.6159 | Val Loss: 11186.7639 | Val NRMSE: 2.5440
Epoch  3 | Train Loss: 10454.4661 | Train NRMSE: 2.4594 | Val Loss: 9904.7841 | Val NRMSE: 2.3938
Epoch  4 | Train Loss: 9259.3533 | Train NRMSE: 2.3145 | Val Loss: 8797.8823 | Val NRMSE: 2.2561
Epoch  5 | Train Loss: 8235.6548 | Train NRMSE: 2.1828 | Val Loss: 7841.6262 | Val NRMSE: 2.1300
Epoch  6 | Train Loss: 7367.5616 | Train NRMSE: 2.0646 | Val Loss: 7016.1467 | Val NRMSE: 2.0148
Epoch  7 | Train Loss: 6606.8722 | Train NRMSE: 1.9551 | Val Loss: 6319.3133 | Val NRMSE: 1.9121
Epoch  8 | Train Loss: 5975.6752 | Train NRMSE: 1.8594 | Val Loss: 5720.0883 | Val NRMSE: 1.8192
Epoch  9 | Train Loss: 5425.3261 | Train NRMSE: 1.7717 | Val Loss: 5216.9809 | Val NRMSE: 1.7373
Epoch 10 | Train Loss: 4982.1816 | Train NRMSE: 1.6978 | Val Loss: 4795.7193 | Val NRMSE